In [13]:
import os
import json
import re
import easyocr
import pandas as pd
import editdistance
from tqdm.notebook import tqdm
import mlflow


In [2]:
VAL_DIR = os.path.join("processed_sroie", "val")
IMG_DIR = os.path.join(VAL_DIR, "img")
ENTITIES_DIR = os.path.join(VAL_DIR, "entities")

reader = easyocr.Reader(['en'], gpu=False)

Using CPU. Note: This module is much faster with a GPU.


In [ ]:
def calculate_cer(predicted, truth):
    if len(truth) == 0:
        return 1.0 if len(predicted) > 0 else 0.0
    # CER = Расстояние Левенштейна / Длина истинной строки
    return editdistance.eval(predicted, truth) / len(truth)

def extract_entities(text_lines):
    full_text = " ".join(text_lines).upper()
    extracted = {
        "company": "UNKNOWN",
        "date": "UNKNOWN",
        "total": "UNKNOWN"
    }
    
    # DD/MM/YYYY   DD-MM-YYYY
    date_match = re.search(r'\b\d{2}[/-]\d{2}[/-]\d{4}\b', full_text)
    if date_match:
        extracted["date"] = date_match.group(0)
    
    # Максимальная из сумм после Total
    amounts = re.findall(r'\b\d+\.\d{2}\b', full_text)
    if amounts:
        amounts_float = [float(a) for a in amounts]
        extracted["total"] = f"{max(amounts_float):.2f}"

    # Ищем первую строку с буквами длинее 3 символов для наименования компании
    for line in text_lines:
        cleaned = line.strip()
        if len(cleaned) > 3 and re.search('[A-Z]', cleaned.upper()):
            extracted["company"] = cleaned
            break
            
    return extracted

In [ ]:
def log_results(metrics):
    macro_f1 = sum(m["F1"] for m in metrics.values()) / 3
    macro_precision = sum(m["Precision"] for m in metrics.values()) / 3
    macro_recall = sum(m["Recall"] for m in metrics.values()) / 3
    macro_cer = sum(m["CER"] for m in metrics.values()) / 3
    
    mlflow.set_tracking_uri("file:./mlruns")
    mlflow.set_experiment("SROIE")
    
    with mlflow.start_run(run_name="Exp2_companies"):
        
        mlflow.log_param("ocr_model", "EasyOCR_Default")
        mlflow.log_param("extractor_type", "Strict_RegEx")
        mlflow.log_param("dataset_split", "Validation")
        
        mlflow.log_metric("Macro_F1", macro_f1)
        mlflow.log_metric("Macro_precision", macro_precision)
        mlflow.log_metric("Macro_recall", macro_recall)
        mlflow.log_metric("Mean_CER", macro_cer)
        
        for field, metrics in metrics.items():
            mlflow.log_metric(f"{field}_F1", metrics["F1"])
            mlflow.log_metric(f"{field}_CER", metrics["CER"])
            mlflow.log_metric(f"{field}_Precision", metrics["Precision"])
            mlflow.log_metric(f"{field}_Recall", metrics["Recall"])
        
        if os.path.exists("experiments.ipynb"):
            mlflow.log_artifact("experiments.ipynb", artifact_path="code")
            

def evaluate():
    results = []
    
    files = [f for f in os.listdir(ENTITIES_DIR) if f.endswith('.txt')]
    
    for filename in tqdm(files, desc="files", position=0):
        file_id = filename.replace('.txt', '')
        img_path = os.path.join(IMG_DIR, f"{file_id}.png")
        txt_path = os.path.join(ENTITIES_DIR, filename)
        
        with open(txt_path, 'r', encoding='utf-8') as f:
            try:
                ground_truth = json.load(f)
            except json.JSONDecodeError:
                continue
                
        ocr_results = reader.readtext(img_path, detail=0) 
        predicted = extract_entities(ocr_results)
         
        for field in ["company", "date", "total"]:
            pred_val = predicted.get(field, "UNKNOWN").upper()
            true_val = ground_truth.get(field, "").upper()
            
            is_exact_match = (pred_val == true_val)
            is_unknown = (pred_val == "UNKNOWN")
            is_empty_truth = (true_val == "")
            
            tp = 1 if is_exact_match and not is_empty_truth else 0
            fp = 1 if not is_exact_match and not is_unknown else 0
            fn = 1 if not is_exact_match and is_unknown and not is_empty_truth else 0
            
            cer = calculate_cer(pred_val, true_val) if not is_unknown else 1.0
            
            results.append({
                "file_id": file_id,
                "field": field,
                "TP": tp,
                "FP": fp,
                "FN": fn,
                "CER": cer
            })
            
    df = pd.DataFrame(results)
    metrics_summary = {}
    
    for field in ["company", "date", "total"]:
        df_field = df[df["field"] == field]
        
        sum_tp = df_field["TP"].sum()
        sum_fp = df_field["FP"].sum()
        sum_fn = df_field["FN"].sum()
        
        precision = sum_tp / (sum_tp + sum_fp) if (sum_tp + sum_fp) > 0 else 0.0
        recall = sum_tp / (sum_tp + sum_fn) if (sum_tp + sum_fn) > 0 else 0.0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        avg_cer = df_field["CER"].mean()
        
        metrics_summary[field] = {
            "Precision": precision,
            "Recall": recall,
            "F1": f1_score,
            "CER": avg_cer
        }
    
    log_results(metrics_summary)
    
    return metrics_summary


In [ ]:
metrics = evaluate()

for field, metrics in metrics.items():
    print(f"Поле: {field.upper()}")
    print(f"  - F1 Score: {metrics['F1']:.4f} (Precision: {metrics['Precision']:.2f}, Recall: {metrics['Recall']:.2f})")
    print(f"  - CER:      {metrics['CER']:.4f}\n")
    
macro_f1 = sum(m["F1"] for m in metrics.values()) / 3
macro_precision = sum(m["Precision"] for m in metrics.values()) / 3
macro_recall = sum(m["Recall"] for m in metrics.values()) / 3
macro_cer = sum(m["CER"] for m in metrics.values()) / 3

print(f"ИТОГОВЫЙ MACRO F1: {macro_f1:.4f}")
print(f"ИТОГОВЫЙ СРЕДНИЙ precision: {macro_precision:.4f}")
print(f"ИТОГОВЫЙ СРЕДНИЙ recall: {macro_recall:.4f}")
print(f"ИТОГОВЫЙ СРЕДНИЙ CER: {macro_cer:.4f}")

log_results(metrics)

files:   0%|          | 0/140 [00:00<?, ?it/s]

c:\VSCode\MTUCI\FSP_full\model\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\VSCode\MTUCI\FSP_full\model\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\VSCode\MTUCI\FSP_full\model\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\VSCode\MTUCI\FSP_full\model\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\VSCode\MTUCI\FSP_full\model\.venv\Lib\site-pa

Поле: COMPANY
  - F1 Score: 0.4783 (Precision: 0.31, Recall: 1.00)
  - CER:      0.4936

Поле: DATE
  - F1 Score: 0.7215 (Precision: 0.94, Recall: 0.59)
  - CER:      0.4043

Поле: TOTAL
  - F1 Score: 0.3626 (Precision: 0.23, Recall: 0.84)
  - CER:      0.4552

ИТОГОВЫЙ MACRO F1: 0.5208
ИТОГОВЫЙ СРЕДНИЙ CER: 0.4510


In [6]:
baseline_metrics = {
    "company": {
         "Precision": 0.31,
            "Recall": 1.0,
            "F1": 0.4783,
            "CER": 0.4936
            }, 
    "date": {
         "Precision": 0.94,
            "Recall": 0.59,
            "F1": 0.7215,
            "CER": 0.4043
            }, 
    "total": {
         "Precision": 0.23,
            "Recall": 0.84,
            "F1": 0.3626,
            "CER": 0.4552
    }}

macro_f1 = sum(m["F1"] for m in baseline_metrics.values()) / 3
macro_precision = sum(m["Precision"] for m in baseline_metrics.values()) / 3
macro_recall = sum(m["Recall"] for m in baseline_metrics.values()) / 3
macro_cer = sum(m["CER"] for m in baseline_metrics.values()) / 3

In [15]:
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("SROIE")

with mlflow.start_run(run_name="Baseline_EasyOCR_RegEx"):
    
    mlflow.log_param("ocr_model", "EasyOCR_Default")
    mlflow.log_param("extractor_type", "Strict_RegEx")
    mlflow.log_param("dataset_split", "Validation")
    
    mlflow.log_metric("Macro_F1", macro_f1)
    mlflow.log_metric("Macro_precision", macro_precision)
    mlflow.log_metric("Macro_recall", macro_recall)
    mlflow.log_metric("Mean_CER", macro_cer)
    
    for field, metrics in baseline_metrics.items():
        mlflow.log_metric(f"{field}_F1", metrics["F1"])
        mlflow.log_metric(f"{field}_CER", metrics["CER"])
        mlflow.log_metric(f"{field}_Precision", metrics["Precision"])
        mlflow.log_metric(f"{field}_Recall", metrics["Recall"])
        
    if os.path.exists("baseline.ipynb"):
        mlflow.log_artifact("baseline.ipynb", artifact_path="code")

2026/03/30 20:18:44 INFO mlflow.tracking.fluent: Experiment with name 'SROIE' does not exist. Creating a new experiment.
